In [ ]:
# Example 1: basic multiple dispatch with @dispatch on functions.
from plum import dispatch

@dispatch
def describe(x: int) -> str:
    return f"int: {x}"

@dispatch
def describe(x: str) -> str:
    return f"str: {x!r}"

@dispatch
def describe(x: float) -> str:
    return f"float: {x:.2f}"

assert describe(7) == "int: 7"
assert describe("hi") == "str: 'hi'"
assert describe(3.14) == "float: 3.14"
print("single-arg dispatch OK")


In [ ]:
# Example 2: multi-argument dispatch — the canonical plum use case.
from plum import dispatch

@dispatch
def combine(a: int, b: int) -> str:
    return f"int+int -> {a + b}"

@dispatch
def combine(a: int, b: str) -> str:
    return f"int+str -> {b * a}"

@dispatch
def combine(a: str, b: str) -> str:
    return f"str+str -> {a + b}"

assert combine(2, 3) == "int+int -> 5"
assert combine(3, "x") == "int+str -> xxx"
assert combine("a", "b") == "str+str -> ab"
print("multi-arg dispatch OK")


In [ ]:
# Example 3: dispatch on user-defined class hierarchy — most-specific wins.
from plum import dispatch

class Animal:
    pass

class Dog(Animal):
    pass

class Puppy(Dog):
    pass

@dispatch
def speak(a: Animal) -> str:
    return "some sound"

@dispatch
def speak(a: Dog) -> str:
    return "woof"

assert speak(Animal()) == "some sound"
assert speak(Dog()) == "woof"
# Puppy is-a Dog: most specific match is Dog.
assert speak(Puppy()) == "woof"
print("hierarchical dispatch OK")


In [ ]:
# Example 4: NotFoundLookupError raised when no method matches.
from plum import dispatch, NotFoundLookupError

@dispatch
def only_for_ints(x: int) -> int:
    return x * 2

assert only_for_ints(5) == 10

raised = False
try:
    only_for_ints("not an int")
except NotFoundLookupError as e:
    raised = True
    print(f"got expected NotFoundLookupError: {type(e).__name__}")

assert raised, "expected NotFoundLookupError for unmatched type"
print("NotFoundLookupError raised correctly")


In [ ]:
# Example 5: dispatch on methods + parametric types via typing.List.
from typing import List

from plum import dispatch

class Processor:
    @dispatch
    def handle(self, x: int) -> str:
        return f"int: {x}"

    @dispatch
    def handle(self, xs: List[int]) -> str:
        return f"List[int] of length {len(xs)}"

    @dispatch
    def handle(self, xs: List[str]) -> str:
        return f"List[str] of length {len(xs)}"

p = Processor()
assert p.handle(42) == "int: 42"
assert p.handle([1, 2, 3]) == "List[int] of length 3"
assert p.handle(["a", "b"]) == "List[str] of length 2"
print("method + parametric dispatch OK")
